# Classification ML Classique sur features TF-IDF

Ce notebook implémente et compare 4 algorithmes classiques de machine learning sur une représentation TF-IDF des avis textuels Yelp :
- Logistic Regression
- Support Vector Machine (LinearSVC)
- Random Forest
- Naive Bayes (MultinomialNB)

**Deux tâches de classification** :
1. **Polarité** (3 classes) : négatif (1-2★), neutre (3★), positif (4-5★)
2. **Score** (5 classes) : prédiction directe des étoiles (1-5)

In [1]:
# Importer les bibliothèques nécessaires
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import joblib
import seaborn as sns
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

## 1. Chargement et Préparation des données

In [2]:
# Définir les chemins
DATA_DIR = '../../data/cleaned'
MODELS_DIR = '../../models/'

# Créer le dossier models s'il n'existe pas
os.makedirs(MODELS_DIR, exist_ok=True)

# Charger les données pré-traitées
print("Chargement des données...")
try:
    df = pd.read_parquet(os.path.join(DATA_DIR, 'reviews_clean.parquet'), engine='fastparquet')
except Exception as e:
    df = pd.read_parquet(os.path.join(DATA_DIR, 'reviews_clean.parquet'))

# On s'assure qu'il n'y a pas de valeurs nulles dans le texte ou la cible
df = df.dropna(subset=['text', 'stars'])

print(f"Dimensions du dataset : {df.shape}")
print(f"Distribution de la cible (stars) :\n{df['stars'].value_counts(normalize=True).sort_index()}")

Chargement des données...
Dimensions du dataset : (999985, 9)
Distribution de la cible (stars) :
stars
1    0.153056
2    0.077629
3    0.098714
4    0.207953
5    0.462647
Name: proportion, dtype: float64


In [ ]:
# Échantillonnage à 10 000 exemples pour la performance locale
SAMPLE_SIZE = 10_000
df = df.sample(n=SAMPLE_SIZE, random_state=42)

# Créer la colonne polarité (3 classes)
df['polarity'] = df['stars'].apply(lambda x: 0 if x <= 2 else (1 if x == 3 else 2))

X = df['text']
y_score = df['stars']
y_polarity = df['polarity']

print(f"Distribution Score :\n{y_score.value_counts().sort_index()}")
print(f"\nDistribution Polarité :\n{y_polarity.value_counts().sort_index()}")

### Split (Train / Val / Test -> 80% / 10% / 10%)

In [ ]:
# Split 80/10/10 — stratifié sur la polarité
X_train, X_temp, y_sc_train, y_sc_temp, y_pol_train, y_pol_temp = train_test_split(
    X, y_score, y_polarity, test_size=0.20, random_state=42, stratify=y_polarity
)
X_val, X_test, y_sc_val, y_sc_test, y_pol_val, y_pol_test = train_test_split(
    X_temp, y_sc_temp, y_pol_temp, test_size=0.50, random_state=42, stratify=y_pol_temp
)

print(f"Taille Train : {X_train.shape[0]} \nTaille Val   : {X_val.shape[0]} \nTaille Test  : {X_test.shape[0]}")

## 2. Vectorisation (TF-IDF)
Nous allons vectoriser notre texte avec tfidf en ajustant uniquement sur l'ensemble d'entraînement pour éviter le data leakage.

In [5]:
print("Vectorisation TF-IDF en cours...")
# On limite à 10000 features pour que les modèles de base tournent de manière fluide.
vectorizer = TfidfVectorizer(max_features=10000, min_df=5, max_df=0.7, ngram_range=(1,2))

X_train_idf = vectorizer.fit_transform(X_train)
X_val_idf = vectorizer.transform(X_val)
X_test_idf = vectorizer.transform(X_test)

print(f"Forme de la matrice TF-IDF d'entraînement : {X_train_idf.shape}")

Vectorisation TF-IDF en cours...
Forme de la matrice TF-IDF d'entraînement : (8000, 10000)


## 3. Entraînement — Tâche 1 : Score (1-5 étoiles)

In [ ]:
def evaluate_model(model, X_train, y_train, X_eval, y_eval):
    """Entraîne le modèle, fait des prédictions et renvoie les métriques (train + val)."""
    model.fit(X_train, y_train)
    
    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_eval)
    
    metrics = {
        'train_acc': accuracy_score(y_train, y_pred_train),
        'val_acc': accuracy_score(y_eval, y_pred_val),
        'val_precision': precision_score(y_eval, y_pred_val, average='macro'),
        'val_recall': recall_score(y_eval, y_pred_val, average='macro'),
        'val_f1': f1_score(y_eval, y_pred_val, average='macro'),
    }
    metrics['overfit_gap'] = metrics['train_acc'] - metrics['val_acc']
    
    return metrics, y_pred_val

def plot_confusion(y_true, y_pred, labels, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=labels, yticklabels=labels)
    plt.title(f'Matrice de confusion — {title}')
    plt.ylabel('Vrai')
    plt.xlabel('Prédit')
    plt.tight_layout()
    plt.show()

In [ ]:
# Initialisation des modèles
models = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42, n_jobs=-1),
    'Linear SVC': LinearSVC(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Naive Bayes': MultinomialNB()
}

results_score = {}
preds_score = {}

for name, model in models.items():
    print(f"\n--- {name} ---")
    metrics, y_pred = evaluate_model(model, X_train_idf, y_sc_train, X_val_idf, y_sc_val)
    results_score[name] = metrics
    preds_score[name] = y_pred
    
    print(f"Train Acc : {metrics['train_acc']:.4f} | Val Acc : {metrics['val_acc']:.4f} | "
          f"Val F1 : {metrics['val_f1']:.4f} | Overfit Gap : {metrics['overfit_gap']:+.4f}")

## 4. Comparaison et Sélection du Meilleur Modèle

In [ ]:
# Tableau comparatif — Score
results_score_df = pd.DataFrame(results_score).T
print("\n=== TABLEAU COMPARATIF — Score (1-5) sur Validation ===")
display(results_score_df.sort_values(by='val_f1', ascending=False))

best_score_name = results_score_df['val_f1'].idxmax()
print(f"\nMeilleur modèle Score : {best_score_name} (F1={results_score_df.loc[best_score_name, 'val_f1']:.4f})")

# Détection d'overfitting
print("\n--- Diagnostic Overfitting ---")
for name, row in results_score_df.iterrows():
    gap = row['overfit_gap']
    flag = " ⚠️ OVERFITTING" if gap > 0.15 else ""
    print(f"{name:25s} : Train {row['train_acc']:.4f} → Val {row['val_acc']:.4f} (gap={gap:+.4f}){flag}")

In [ ]:
# Matrice de confusion du meilleur modèle — Score
score_labels = ['1★', '2★', '3★', '4★', '5★']
plot_confusion(y_sc_val, preds_score[best_score_name], score_labels, f'{best_score_name} — Score')

print(f"\nRapport de classification — {best_score_name} (Score) :\n")
print(classification_report(y_sc_val, preds_score[best_score_name], target_names=score_labels))

## 4. Entraînement — Tâche 2 : Polarité (3 classes)

In [ ]:
# Ré-initialiser les modèles pour la polarité
models_pol = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42, n_jobs=-1),
    'Linear SVC': LinearSVC(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Naive Bayes': MultinomialNB()
}

results_pol = {}
preds_pol = {}

for name, model in models_pol.items():
    print(f"\n--- {name} ---")
    metrics, y_pred = evaluate_model(model, X_train_idf, y_pol_train, X_val_idf, y_pol_val)
    results_pol[name] = metrics
    preds_pol[name] = y_pred
    
    print(f"Train Acc : {metrics['train_acc']:.4f} | Val Acc : {metrics['val_acc']:.4f} | "
          f"Val F1 : {metrics['val_f1']:.4f} | Overfit Gap : {metrics['overfit_gap']:+.4f}")

In [ ]:
# Tableau comparatif — Polarité
results_pol_df = pd.DataFrame(results_pol).T
print("\n=== TABLEAU COMPARATIF — Polarité (3 classes) sur Validation ===")
display(results_pol_df.sort_values(by='val_f1', ascending=False))

best_pol_name = results_pol_df['val_f1'].idxmax()
print(f"\nMeilleur modèle Polarité : {best_pol_name} (F1={results_pol_df.loc[best_pol_name, 'val_f1']:.4f})")

# Détection d'overfitting
print("\n--- Diagnostic Overfitting ---")
for name, row in results_pol_df.iterrows():
    gap = row['overfit_gap']
    flag = " ⚠️ OVERFITTING" if gap > 0.15 else ""
    print(f"{name:25s} : Train {row['train_acc']:.4f} → Val {row['val_acc']:.4f} (gap={gap:+.4f}){flag}")

# Matrice de confusion
pol_labels = ['Négatif', 'Neutre', 'Positif']
plot_confusion(y_pol_val, preds_pol[best_pol_name], pol_labels, f'{best_pol_name} — Polarité')

print(f"\nRapport de classification — {best_pol_name} (Polarité) :\n")
print(classification_report(y_pol_val, preds_pol[best_pol_name], target_names=pol_labels))

## 5. Test Final et Sauvegarde

In [ ]:
# Évaluation finale sur le test set
best_model_score = models[best_score_name]
best_model_pol = models_pol[best_pol_name]

test_pred_sc = best_model_score.predict(X_test_idf)
test_pred_pol = best_model_pol.predict(X_test_idf)

test_f1_sc = f1_score(y_sc_test, test_pred_sc, average='macro')
test_f1_pol = f1_score(y_pol_test, test_pred_pol, average='macro')
test_acc_sc = accuracy_score(y_sc_test, test_pred_sc)
test_acc_pol = accuracy_score(y_pol_test, test_pred_pol)

print(f"=== PERFORMANCE FINALE SUR TEST ===")
print(f"Score    ({best_score_name:25s}) : Acc={test_acc_sc:.4f} | F1 Macro={test_f1_sc:.4f}")
print(f"Polarité ({best_pol_name:25s}) : Acc={test_acc_pol:.4f} | F1 Macro={test_f1_pol:.4f}")

# Sauvegarder les modèles
model_path_sc = os.path.join(MODELS_DIR, 'best_tfidf_classifier.pkl')
model_path_pol = os.path.join(MODELS_DIR, 'best_tfidf_polarity.pkl')
vectorizer_path = os.path.join(MODELS_DIR, 'tfidf_vectorizer.pkl')

joblib.dump(best_model_score, model_path_sc)
joblib.dump(best_model_pol, model_path_pol)
joblib.dump(vectorizer, vectorizer_path)

print(f"\nModèles sauvegardés dans {MODELS_DIR}")